<a href="https://colab.research.google.com/github/avirooppal/Remedy-LLM/blob/main/RemedyLLMTrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
df =  pd.read_csv('/content/Home Remedies.csv')
df.head()
print(len(df))
df.columns

115


Index(['Name of Item', 'Health Issue', 'Home Remedy', 'Yogasan'], dtype='object')

In [2]:
pip install -q transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.6 MB/s eta 0:00:00


In [13]:
import os
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "/content/Outputs/remedies-slm"

USER_TEMPLATES = [
    "I have {issue}. I only have {item} at home. Suggest a remedy and a yoga pose.",
    "What can I do for {issue} using {item}? Also suggest a yoga practice.",
    "Suggest a home remedy for {issue} using {item} and a suitable yogasan.",
    "I am suffering from {issue}. Can {item} help? Also suggest yoga."
]

def expand_to_chat(sample):
    outputs = []
    for template in USER_TEMPLATES:
        user_msg = template.format(
            issue=sample["Health Issue"],
            item=sample["Name of Item"]
        )
        assistant_msg = f"""Home Remedy:
{sample['Home Remedy']}

Yogasan:
{sample['Yogasan']}"""

        text = f"""<|user|>
{user_msg}

<|assistant|>
{assistant_msg}"""

        outputs.append({"text": text})
    return outputs

def main():
    print("Loading CSV dataset...")
    dataset = load_dataset("csv", data_files="/content/Home Remedies.csv")["train"]
    print("Original size:", len(dataset))
    print("Columns:", dataset.column_names)

    print("Expanding dataset...")
    expanded = []
    for sample in dataset:
        expanded.extend(expand_to_chat(sample))
    dataset = Dataset.from_list(expanded)
    print("Expanded size:", len(dataset))

    print(f"Loading {MODEL_NAME}...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float32,  # ✅ float32 avoids bf16/fp16 scaler conflict
        bnb_4bit_use_double_quant=True
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    model.config.use_cache = False
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=12,
        per_device_train_batch_size=4,         # ✅ Reduced since we dropped fp16
        gradient_accumulation_steps=2,          # ✅ Compensates for smaller batch
        learning_rate=5e-4,
        logging_steps=5,
        save_steps=100,
        save_total_limit=1,
        fp16=False,                             # ✅ Disabled — root cause of the error
        bf16=False,                             # ✅ Disabled
        optim="adamw_8bit",                     # ✅ Memory-efficient optimizer from bitsandbytes
        dataset_text_field="text",
        eval_strategy="no",
        max_length=512,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        processing_class=tokenizer,
    )

    print("Starting training...")
    trainer.train()

    print("Saving model...")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Done! Model saved to:", OUTPUT_DIR)

if __name__ == "__main__":
    main()

Loading CSV dataset...
Original size: 115
Columns: ['Name of Item', 'Health Issue', 'Home Remedy', 'Yogasan']
Expanding dataset...
Expanded size: 460
Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


Adding EOS to train dataset:   0%|          | 0/460 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/460 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Step,Training Loss
5,2.517117
10,2.145433
15,1.834238
20,1.636655
25,1.482997
30,1.618704
35,1.455515
40,1.442528
45,1.267714
50,1.235591


Saving model...
Done! Model saved to: /content/Outputs/remedies-slm


In [9]:
import trl, transformers, peft
print("TRL:", trl.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

TRL: 1.2.0
Transformers: 5.0.0
PEFT: 0.18.1


In [11]:
import inspect
from trl import SFTConfig
print(inspect.signature(SFTConfig.__init__))

(self, output_dir: str | None = None, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: transformers.trainer_utils.IntervalStrategy | str = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, gradient_accumulation_steps: int = 1, eval_accumulation_steps: int | None = None, eval_delay: float = 0, torch_empty_cache_steps: int | None = None, learning_rate: float = 2e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_ratio: float | None = None, warmup_steps: float = 0, log_level: str = 'passive', log_level_replica: str = 'warning', log_on_each_node: bool = True, logging_dir: str | None = None, logging_strategy: transfor